# Getting Started with Goqu

This notebook introduces Goqu's core features: building quantum circuits,
visualizing them as inline SVG diagrams, and running simulations.

**Requirements:** [gonb](https://github.com/janpfeifer/gonb) Go Jupyter kernel.

```
go install github.com/janpfeifer/gonb@latest && gonb --install
```

In [1]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/statevector"
)

In [2]:
func buildBell() *ir.Circuit {
	c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
	return c
}

func buildGHZ() *ir.Circuit {
	c, _ := builder.New("ghz", 3).H(0).CNOT(0, 1).CNOT(0, 2).Build()
	return c
}

## Building a Bell State

A Bell state is the simplest entangled circuit: a Hadamard gate followed by a CNOT.

In [3]:
%%
fmt.Println(draw.String(buildBell()))

q0: -H---@--
         |
q1: -----X--



## Inline SVG Visualization

`draw.SVG()` returns an SVG string that `gonbui.DisplaySvg()` renders inline.

In [4]:
%%
gonbui.DisplaySvg(draw.SVG(buildBell()))

q0 
 
 q1 
 
 H

### Dark Theme

Pass `draw.WithStyle(draw.DarkStyle())` for a dark background.

In [5]:
%%
gonbui.DisplaySvg(draw.SVG(buildBell(), draw.WithStyle(draw.DarkStyle())))

q0 
 
 q1 
 
 H

## Statevector Simulation

Use `statevector.New()` to create a simulator, then `Evolve()` to apply the
circuit without measurement collapse.

In [6]:
%%
sim := statevector.New(2)
sim.Evolve(buildBell())
for i, amp := range sim.StateVector() {
	if p := real(amp)*real(amp) + imag(amp)*imag(amp); p > 1e-10 {
		fmt.Printf("|%02b> amplitude: %.4f  probability: %.1f\n", i, amp, math.Round(p*10)/10)
	}
}

|00> amplitude: (0.7071+0.0000i)  probability: 0.5
|11> amplitude: (0.7071+0.0000i)  probability: 0.5


## Shot-Based Sampling

Add measurements and sample 1000 shots. The circuit needs `MeasureAll()` for `Run()`.

In [7]:
%%
bellM, _ := builder.New("bell-measured", 2).H(0).CNOT(0, 1).MeasureAll().Build()
sim2 := statevector.New(2)
counts, _ := sim2.Run(bellM, 1000)
for state, n := range counts {
	fmt.Printf("|%s>: %d\n", state, n)
}

|00>: 519
|11>: 481


## GHZ State

A GHZ state generalizes entanglement to 3+ qubits: H on qubit 0, then CNOT to each subsequent qubit.

In [8]:
%%
gonbui.DisplaySvg(draw.SVG(buildGHZ()))

q0 
 
 q1 
 
 q2 
 
 H

In [9]:
%%
simGHZ := statevector.New(3)
simGHZ.Evolve(buildGHZ())
for i, amp := range simGHZ.StateVector() {
	if p := real(amp)*real(amp) + imag(amp)*imag(amp); p > 1e-10 {
		fmt.Printf("|%03b> probability: %.2f\n", i, p)
	}
}

|000> probability: 0.50
|111> probability: 0.50


## Circuit Statistics

`Stats()` returns depth, gate count, two-qubit gates, and more.

In [10]:
%%
s := buildGHZ().Stats()
fmt.Printf("Depth:          %d\n", s.Depth)
fmt.Printf("Gate count:     %d\n", s.GateCount)
fmt.Printf("Two-qubit gates: %d\n", s.TwoQubitGates)

Depth:          3
Gate count:     3
Two-qubit gates: 2
